# Plant Incident Analytics Pipeline

Attach a schema-enabled default Fabric Lakehouse, upload the configuration to `Files/configuration/` and the sample source to `Files/data/`, then run all cells. See `docs/setup.md`.

The final optional cell exports the reporting tables for the supplied Power BI project. Notebook outputs and workspace identifiers are intentionally omitted.

In [ ]:
# Read and validate configuration
import re
import pandas as pd
import uuid
from fnmatch import fnmatchcase

metadata_path = "/lakehouse/default/Files/configuration/incident_pipeline_config.xlsx"

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Read all metadata.
metadata = pd.read_excel(metadata_path, sheet_name=None, dtype=str, keep_default_na=False)
required_sheets = {"pipeline_config", "category_mapping"}
if not required_sheets.issubset(metadata):
    raise ValueError(f"Metadata workbook needs these sheets: {sorted(required_sheets)}")

settings_pdf = metadata["pipeline_config"]
if not {"setting", "value"}.issubset(settings_pdf.columns):
    raise ValueError("pipeline_config needs setting and value columns.")
settings_pdf = settings_pdf[["setting", "value"]].apply(lambda column: column.str.strip())
if settings_pdf.eq("").any().any() or settings_pdf["setting"].duplicated().any():
    raise ValueError("Configuration has empty values or duplicate setting names.")
config = dict(zip(settings_pdf["setting"], settings_pdf["value"]))
required_settings = {
    "source_folder", "archive_folder", "failed_folder", "file_pattern", "sheet_name",
    "plant_column", "month_column", "audit_table", "bronze_table", "plant_table",
    "fact_table"
}
if not required_settings.issubset(config):
    raise ValueError(f"Missing settings: {sorted(required_settings - config.keys())}")

# Paths and table names are checked before they are used in file moves or SQL.
folders = [config[key].rstrip("/") for key in ["source_folder", "archive_folder", "failed_folder"]]
for folder in folders:
    if not folder.startswith("Files/") or any(part in {"", ".", ".."} for part in folder.split("/")) or "\\" in folder:
        raise ValueError("Folders must be paths beneath Files in the default Lakehouse.")
for i, folder in enumerate(folders):
    for other in folders[i + 1:]:
        if folder == other or folder.startswith(other + "/") or other.startswith(folder + "/"):
            raise ValueError("Source, archive and failed folders must be separate.")
if any(metadata_path.startswith(f"/lakehouse/default/{folder}/") for folder in folders):
    raise ValueError("Keep the metadata workbook outside the source, archive and failed folders.")
source_folder, archive_folder, failed_folder = folders

table_names = [config[key] for key in ["audit_table", "bronze_table", "plant_table", "fact_table"]]
if any(not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", name) for name in table_names):
    raise ValueError("Table names must use schema.table with letters, numbers and underscores.")
if len({name.lower() for name in table_names}) != 4:
    raise ValueError("Audit, Bronze, plant and fact tables must have different names.")
audit_table, bronze_table, plant_table, fact_table = table_names
plant_column, month_column = config["plant_column"], config["month_column"]
if plant_column == month_column:
    raise ValueError("Plant and month source columns must be different.")

mapping_pdf = metadata["category_mapping"]
mapping_columns = ["source_category", "classification", "code_group"]
if not set(mapping_columns).issubset(mapping_pdf.columns):
    raise ValueError("category_mapping needs source_category, classification and code_group.")
mapping_pdf = mapping_pdf[mapping_columns].apply(lambda column: column.str.strip())
if mapping_pdf.empty or mapping_pdf.eq("").any().any() or mapping_pdf["source_category"].duplicated().any():
    raise ValueError("Category mapping is empty, incomplete or contains duplicate categories.")

category_mapping_df = spark.createDataFrame(mapping_pdf)
category_mapping_df.createOrReplaceTempView("incident_category_mapping")
for folder in folders:
    _ = notebookutils.fs.mkdirs(folder)

print("Configuration and category mappings validated.")
display(settings_pdf)
display(mapping_pdf)


In [ ]:
# Create missing schemas and tables
for schema_name in sorted({name.split(".")[0] for name in table_names}):
    _ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

# Create audit table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    pipeline_run_id STRING NOT NULL,
    source_file_name STRING,
    started_at_utc TIMESTAMP,
    completed_at_utc TIMESTAMP,
    status STRING,
    current_stage STRING,
    source_row_count BIGINT,
    bronze_row_count BIGINT,
    silver_row_count BIGINT,
    fact_row_count BIGINT,
    error_message STRING
)
USING DELTA
""")

# Create Bronze table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_table} (
    source_row_number INT,
    plant_raw STRING,
    month_raw STRING,
    source_category STRING,
    incident_count_raw STRING,
    load_run_id STRING,
    source_file_name STRING,
    source_file_path STRING,
    loaded_at_utc TIMESTAMP
)
USING DELTA
""")

# Create plant dimension
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {plant_table} (
    plant_code STRING NOT NULL,
    is_active BOOLEAN NOT NULL
)
USING DELTA
""")

# Create fact table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fact_table} (
    plant_code STRING NOT NULL,
    reporting_month DATE NOT NULL,
    source_category STRING NOT NULL,
    classification STRING NOT NULL,
    code_group STRING NOT NULL,
    incident_count BIGINT NOT NULL,
    pipeline_run_id STRING,
    source_file_name STRING,
    source_row_number BIGINT,
    loaded_at_utc TIMESTAMP
)
USING DELTA
""")

In [ ]:
# Read the Excel file and check its headers
def read_source(source_path):
    sheet_pdf = pd.read_excel(
        source_path,
        sheet_name=config["sheet_name"],
        header=None,
        dtype=object,
        keep_default_na=False
    )

    if sheet_pdf.empty:
        raise ValueError("The source worksheet is empty.")

    headers = sheet_pdf.iloc[0].tolist()
    if any(not isinstance(header, str) or not header.strip() for header in headers):
        raise ValueError("Missing or invalid column headers.")
    if len(headers) != len(set(headers)):
        raise ValueError("Duplicate column headers.")
    if not {plant_column, month_column}.issubset(headers):
        raise ValueError(f"Source must contain {plant_column} and {month_column}.")

    incident_columns = [column for column in headers if column not in [plant_column, month_column]]
    if not incident_columns:
        raise ValueError("No incident columns found.")

    raw_pdf = sheet_pdf.iloc[1:].copy()
    raw_pdf.columns = headers
    raw_pdf = raw_pdf.loc[~raw_pdf.eq("").all(axis=1)]
    if raw_pdf.empty:
        raise ValueError("The source worksheet has no data records.")

    print(f"Source rows: {len(raw_pdf)}; incident columns: {len(incident_columns)}")
    return raw_pdf, incident_columns


In [ ]:
# Unpivot and load Bronze using SQL
def load_bronze(raw_pdf, incident_columns, run):
    column_lookup = {
        header: f"column_{position}"
        for position, header in enumerate(raw_pdf.columns)
    }
    rows = [
        (
            int(index) + 1,
            *[None if pd.isna(value) or value == "" else str(value) for value in row]
        )
        for index, row in raw_pdf.iterrows()
    ]
    schema = "source_row_number INT" + "".join(
        f", {column} STRING" for column in column_lookup.values()
    )
    spark.createDataFrame(rows, schema).createOrReplaceTempView("excel_source")

    stack_columns = ",\n                    ".join(
        f":category_{position}, {column_lookup[header]}"
        for position, header in enumerate(incident_columns)
    )
    parameters = {
        f"category_{position}": header
        for position, header in enumerate(incident_columns)
    }
    parameters.update({
        "run_id": run["pipeline_run_id"],
        "file_name": run["source_file_name"],
        "file_path": run["source_path"]
    })

    bronze_df = spark.sql(f"""
        SELECT
            source_row_number,
            {column_lookup[plant_column]} AS plant_raw,
            {column_lookup[month_column]} AS month_raw,
            source_category,
            incident_count_raw,
            :run_id AS load_run_id,
            :file_name AS source_file_name,
            :file_path AS source_file_path,
            current_timestamp() AS loaded_at_utc
        FROM excel_source
        LATERAL VIEW STACK(
            {len(incident_columns)},
            {stack_columns}
        ) unpivoted AS source_category, incident_count_raw
    """, args=parameters)
    bronze_df.createOrReplaceTempView("bronze_load")

    spark.sql(f"""
        INSERT OVERWRITE TABLE {bronze_table}
        SELECT
            source_row_number,
            plant_raw,
            month_raw,
            source_category,
            incident_count_raw,
            load_run_id,
            source_file_name,
            source_file_path,
            loaded_at_utc
        FROM bronze_load
    """)

    bronze_df = spark.sql(f"""
        SELECT *
        FROM {bronze_table}
        WHERE load_run_id = :run_id
    """, args={"run_id": run["pipeline_run_id"]})
    bronze_df.createOrReplaceTempView("bronze_current")

    bronze_row_count = spark.sql("""
        SELECT COUNT(*) AS row_count
        FROM bronze_current
    """).first()["row_count"]

    if bronze_row_count != len(raw_pdf) * len(incident_columns):
        raise ValueError("Bronze row count does not match the expected unpivot.")

    print(f"Bronze loaded: {bronze_row_count} rows")
    return bronze_df, bronze_row_count


In [ ]:
# Clean and validate Silver using SQL
def validate_silver(bronze_df):
    bronze_df.createOrReplaceTempView("bronze_current")

    silver_df = spark.sql("""
        SELECT
            *,
            UPPER(TRIM(plant_raw)) AS plant_code,
            TRIM(source_category) AS category_clean,
            TRY_CAST(TRIM(month_raw) AS DATE) AS reporting_month,
            TRY_CAST(
                TRY_CAST(TRIM(incident_count_raw) AS DECIMAL(38, 0))
                AS BIGINT
            ) AS incident_count
        FROM bronze_current
    """)
    silver_df.createOrReplaceTempView("silver_checked")

    results = spark.sql("""
        SELECT
            SUM(CASE
                WHEN plant_code IS NULL OR plant_code = ''
                THEN 1 ELSE 0
            END) AS missing_plants,

            SUM(CASE
                WHEN category_clean IS NULL OR category_clean = ''
                THEN 1 ELSE 0
            END) AS missing_categories,

            SUM(CASE
                WHEN reporting_month IS NULL
                  OR DAYOFMONTH(reporting_month) <> 1
                  OR NOT COALESCE(
                      TRIM(month_raw) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}( 00:00:00([.]0+)?)?$',
                      FALSE
                  )
                THEN 1 ELSE 0
            END) AS invalid_months,

            SUM(CASE
                WHEN incident_count IS NULL
                  OR NOT COALESCE(
                      TRIM(incident_count_raw) RLIKE '^[+]?[0-9]+([.]0+)?$',
                      FALSE
                  )
                THEN 1 ELSE 0
            END) AS invalid_counts
        FROM silver_checked
    """).first().asDict()

    print(f"Silver checks: {results}")
    if any(count for count in results.values()):
        raise ValueError(f"Silver data quality checks failed: {results}")

    duplicates = spark.sql("""
        SELECT
            plant_code,
            reporting_month,
            category_clean,
            COUNT(*) AS record_count
        FROM silver_checked
        GROUP BY plant_code, reporting_month, category_clean
        HAVING COUNT(*) > 1
    """)

    if not duplicates.isEmpty():
        display(duplicates.limit(20))
        raise ValueError("Duplicate plant-month-category combinations.")

    print("Silver checks passed; no duplicate keys.")
    return silver_df


In [ ]:
# Apply the Excel category mappings using SQL
def apply_category_mapping(silver_df):
    silver_df.createOrReplaceTempView("silver_checked")

    unmapped = spark.sql("""
        SELECT DISTINCT source.category_clean AS source_category
        FROM silver_checked AS source
        LEFT ANTI JOIN incident_category_mapping AS mapping
            ON source.category_clean = mapping.source_category
    """)

    if not unmapped.isEmpty():
        names = [row.source_category for row in unmapped.limit(20).collect()]
        raise ValueError(f"Add missing categories to the metadata workbook: {names}")

    silver_df = spark.sql("""
        SELECT
            source.plant_code,
            source.reporting_month,
            source.category_clean AS source_category,
            mapping.classification,
            mapping.code_group,
            source.incident_count,
            source.load_run_id AS pipeline_run_id,
            source.source_file_name,
            CAST(source.source_row_number AS BIGINT) AS source_row_number,
            source.loaded_at_utc
        FROM silver_checked AS source
        LEFT JOIN incident_category_mapping AS mapping
            ON source.category_clean = mapping.source_category
    """)
    silver_df.createOrReplaceTempView("silver_incidents")

    silver_row_count = spark.sql("""
        SELECT COUNT(*) AS row_count
        FROM silver_incidents
    """).first()["row_count"]

    print(f"Category mapping applied: {silver_row_count} Silver rows")
    display(spark.sql("""
        SELECT
            classification,
            code_group,
            SUM(incident_count) AS total_incidents
        FROM silver_incidents
        GROUP BY classification, code_group
        ORDER BY classification, code_group
    """))
    return silver_df, silver_row_count


In [ ]:
# Load the plant dimension using SQL
def load_plants(silver_df):
    silver_df.createOrReplaceTempView("silver_incidents")

    spark.sql(f"""
        MERGE INTO {plant_table} AS target
        USING (
            SELECT DISTINCT plant_code
            FROM silver_incidents
        ) AS source
        ON target.plant_code = source.plant_code

        WHEN NOT MATCHED THEN
            INSERT (plant_code, is_active)
            VALUES (source.plant_code, TRUE)
    """)

    print("Plant dimension merge completed.")


In [ ]:
# Load new and changed fact records using SQL
def load_fact(silver_df):
    silver_df.createOrReplaceTempView("silver_incidents")

    spark.sql(f"""
        MERGE INTO {fact_table} AS target
        USING silver_incidents AS source
        ON  target.plant_code = source.plant_code
        AND target.reporting_month = source.reporting_month
        AND target.source_category = source.source_category

        WHEN MATCHED AND (
            NOT (target.incident_count <=> source.incident_count)
            OR NOT (target.classification <=> source.classification)
            OR NOT (target.code_group <=> source.code_group)
        )
        THEN UPDATE SET *

        WHEN NOT MATCHED
        THEN INSERT *
    """)

    print("Fact merge completed: new keys inserted, changed business values updated.")


In [ ]:
# Reconcile the loaded data using SQL
def reconcile_fact(silver_df):
    silver_df.createOrReplaceTempView("silver_incidents")

    differences = spark.sql(f"""
        WITH incoming AS (
            SELECT
                plant_code, reporting_month, source_category,
                classification, code_group, incident_count
            FROM silver_incidents
        ),
        stored AS (
            SELECT
                target.plant_code, target.reporting_month, target.source_category,
                target.classification, target.code_group, target.incident_count
            FROM {fact_table} AS target
            INNER JOIN incoming AS source
                ON  target.plant_code = source.plant_code
                AND target.reporting_month = source.reporting_month
                AND target.source_category = source.source_category
        ),
        missing_or_changed AS (
            SELECT * FROM incoming
            EXCEPT ALL
            SELECT * FROM stored
        ),
        extra_or_changed AS (
            SELECT * FROM stored
            EXCEPT ALL
            SELECT * FROM incoming
        )
        SELECT * FROM missing_or_changed
        UNION ALL
        SELECT * FROM extra_or_changed
    """)

    if not differences.isEmpty():
        display(differences.limit(20))
        raise ValueError("Stored fact data does not match the validated input.")

    missing_plants = spark.sql(f"""
        SELECT DISTINCT fact.plant_code
        FROM {fact_table} AS fact
        LEFT ANTI JOIN {plant_table} AS plant
            ON fact.plant_code = plant.plant_code
    """)
    if not missing_plants.isEmpty():
        raise ValueError("Fact records have no matching plant dimension.")

    fact_row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {fact_table}
    """).first()["row_count"]

    print(f"Reconciliation passed; total fact-table rows: {fact_row_count}")
    return fact_row_count


In [ ]:
# Archive a successful source file
def archive_source(run):
    source_file_path = run["source_file_path"]
    archive_path = run["archive_path"]
    moved = notebookutils.fs.mv(source_file_path, archive_path, create_path=True, overwrite=False)
    if not moved or notebookutils.fs.exists(source_file_path) or not notebookutils.fs.exists(archive_path):
        raise RuntimeError("Gold loaded, but the archive move could not be verified.")
    print(f"Archived: {archive_path}")


In [ ]:
# Update the audit table
def update_audit(run_id, stage, status="IN_PROGRESS", source_rows=None,
                 bronze_rows=None, silver_rows=None, fact_rows=None,
                 error_message=None, complete=False):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
            current_stage = :stage,
            status = :status,
            source_row_count = COALESCE(:source_rows, source_row_count),
            bronze_row_count = COALESCE(:bronze_rows, bronze_row_count),
            silver_row_count = COALESCE(:silver_rows, silver_row_count),
            fact_row_count = COALESCE(:fact_rows, fact_row_count),
            error_message = :error_message,
            completed_at_utc = CASE
                WHEN :complete THEN current_timestamp()
                ELSE completed_at_utc
            END
        WHERE pipeline_run_id = :run_id
    """, args={
        "run_id": run_id,
        "stage": stage,
        "status": status,
        "source_rows": source_rows,
        "bronze_rows": bronze_rows,
        "silver_rows": silver_rows,
        "fact_rows": fact_rows,
        "error_message": error_message,
        "complete": complete
    })


In [ ]:
#load all files.
source_files = sorted(
    file.name for file in notebookutils.fs.ls(source_folder)
    if file.isFile and not file.name.startswith("~$")
    and fnmatchcase(file.name, config["file_pattern"])
    and file.name.lower().endswith(".xlsx")
)
print(f"Matching files: {len(source_files)}")

if not source_files:
    _ = spark.sql(f"""
        INSERT INTO {audit_table} (
            pipeline_run_id, started_at_utc, completed_at_utc, status, current_stage
        )
        VALUES (
            :run_id, current_timestamp(), current_timestamp(), 'SKIPPED', 'no_files'
        )
    """, args={"run_id": str(uuid.uuid4())})
    print("No matching files to load. Audit status: SKIPPED.")

for source_file_name in source_files:
    pipeline_run_id = str(uuid.uuid4())
    source_file_path = f"{source_folder}/{source_file_name}"
    source_path = f"/lakehouse/default/{source_file_path}"
    archive_path = f"{archive_folder}/{pipeline_run_id}/{source_file_name}"
    failed_path = f"{failed_folder}/{pipeline_run_id}/{source_file_name}"
    run = {
        "pipeline_run_id": pipeline_run_id,
        "source_file_name": source_file_name,
        "source_file_path": source_file_path,
        "source_path": source_path,
        "archive_path": archive_path
    }
    stage = "start"
    audit_started = False
    archive_attempted = False
    print(f"Starting {source_file_name}; run ID: {pipeline_run_id}")

    try:
        _ = spark.sql(f"""
            INSERT INTO {audit_table} (
                pipeline_run_id, source_file_name, started_at_utc, status, current_stage
            )
            VALUES (
                :run_id, :file_name, current_timestamp(), 'STARTED', 'start'
            )
        """, args={"run_id": pipeline_run_id, "file_name": source_file_name})
        audit_started = True

        stage = "source_validation"
        update_audit(pipeline_run_id, stage)
        raw_pdf, incident_columns = read_source(source_path)
        update_audit(pipeline_run_id, stage, source_rows=len(raw_pdf))

        stage = "bronze"
        update_audit(pipeline_run_id, stage)
        bronze_df, bronze_row_count = load_bronze(raw_pdf, incident_columns, run)
        update_audit(pipeline_run_id, stage, bronze_rows=bronze_row_count)

        stage = "silver_validation"
        update_audit(pipeline_run_id, stage)
        silver_df = validate_silver(bronze_df)

        stage = "category_mapping"
        update_audit(pipeline_run_id, stage)
        silver_df, silver_row_count = apply_category_mapping(silver_df)
        update_audit(pipeline_run_id, stage, silver_rows=silver_row_count)

        stage = "plant_load"
        update_audit(pipeline_run_id, stage)
        load_plants(silver_df)

        stage = "fact_load"
        update_audit(pipeline_run_id, stage)
        load_fact(silver_df)

        stage = "reconciliation"
        update_audit(pipeline_run_id, stage)
        fact_row_count = reconcile_fact(silver_df)

        stage = "archive"
        update_audit(pipeline_run_id, stage)
        archive_attempted = True
        archive_source(run)

        stage = "audit_complete"
        update_audit(pipeline_run_id, "complete", status="SUCCESS", fact_rows=fact_row_count, complete=True)
        print(f"SUCCESS: {source_file_name}")

    except Exception as error:
        error_message = str(error)
        try:
            if archive_attempted and notebookutils.fs.exists(archive_path):
                error_message += f" | Archive exists at {archive_path}; check for a remaining source copy."
            elif notebookutils.fs.exists(source_file_path):
                moved = notebookutils.fs.mv(source_file_path, failed_path, create_path=True, overwrite=False)
                if not moved or notebookutils.fs.exists(source_file_path) or not notebookutils.fs.exists(failed_path):
                    raise RuntimeError("The failed-file move could not be verified.")
                error_message += f" | File moved to {failed_path}"
        except Exception as move_error:
            error_message += f" | File move error: {move_error}"
        if audit_started:
            try:
                update_audit(pipeline_run_id, stage, status="FAILED", error_message=error_message, complete=True)
            except Exception as audit_error:
                error_message += f" | Audit update error: {audit_error}"
        print(f"FAILED: {source_file_name}; stage: {stage}; {error_message}")
        raise RuntimeError(f"File {source_file_name}; run {pipeline_run_id}; stage {stage}: {error_message}") from error


In [ ]:
df = spark.sql(f"SELECT * FROM {audit_table} ORDER BY started_at_utc DESC LIMIT 1000")
display(df)

## Export the reporting handoff

Run this after ingestion succeeds. These Excel exports feed the Power BI Import model. The Fabric Gold tables retain lineage columns; this export selects only the reporting fields.

In [ ]:
from pathlib import Path

reporting_folder = "Files/reporting"
if any(reporting_folder == folder or reporting_folder.startswith(folder + "/") or folder.startswith(reporting_folder + "/") for folder in folders):
    raise ValueError("Keep the reporting export folder separate from processing folders.")
_ = notebookutils.fs.mkdirs(reporting_folder)
reporting_path = Path("/lakehouse/default") / reporting_folder

spark.table(plant_table).select("plant_code", "is_active").orderBy("plant_code").toPandas().to_excel(
    reporting_path / "dim_plant.xlsx", sheet_name="sheet1", index=False
)
spark.table(fact_table).select(
    "plant_code", "reporting_month", "source_category", "classification", "code_group", "incident_count"
).orderBy("plant_code", "reporting_month", "source_category").toPandas().to_excel(
    reporting_path / "fact_monthly_incidents.xlsx", sheet_name="sheet1", index=False
)
print("Reporting exports written to Files/reporting. Download both files for Power BI.")
